# FundFirst

This notebook presents the end-to-end FundFirst proof-of-concept workflow:

1. Environment and project-folder setup
2. Cleaning of the four source datasets
3. Borough-year dataset merging and validation
4. Transparent TSM calculation and initial labels
5. Sensitivity calibration of the Achievable threshold using 2014–2022 only
6. Final three-class label generation and baseline self-agreement check
7. Chronological train, validation, and held-out test split
8. Logistic Regression and Random Forest training
9. 2023 validation checkpoint and frozen model settings
10. Final 2024–2025 held-out evaluation and model selection
11. Local and global SHAP explainability
12. Income-stratified fairness audit
13. Model card generation
14. Model serialization and reload verification
15. Manual inference test for a new scenario
16. Final project summary

**Evaluation boundary:** the 2024–2025 held-out test period is used once for final comparative evaluation. No further model tuning is performed after viewing these results.

**Interpretation boundary:** model performance measures agreement with the project’s rule-generated feasibility labels. It is not evidence of real-world mortgage approval or affordability accuracy.

**Data sources:** UK House Price Index monthly average-price series, ASHE workplace-based earnings, ONS Household Saving Ratio, and Bank of England Bank Rate.

**Final project configuration:**
- Dataset: **72 borough-year observations** across six South West London boroughs, 2014–2025.
- TSM label rule: **Achievable ≤ 72 months**, **Stretch > 72 and ≤ 108 months**, **Unfeasible > 108 months**.
- Initial model training: 2014–2022; validation checkpoint: 2023; final refit: 2014–2023; held-out test: 2024–2025.
- Selected prototype model: **Logistic Regression**.
- Held-out test performance: Logistic Regression **accuracy = 0.833, macro F1 = 0.778**; Random Forest **accuracy = 0.583, macro F1 = 0.444**.
- Explainability: local SHAP explanations are generated for all 12 held-out predictions, with a global contribution summary for model-level interpretation.
- Fairness audit: income-stratified borough groups are compared using a predefined 10 percentage-point flag threshold. The current held-out audit flags the overall accuracy gap and Stretch precision gap; precision is left undefined where a group has no predictions for a class.

## 1. Environment setup

In [4]:
import shap
print("SHAP version:", shap.__version__)

/private/tmp/fundfirst_submission_py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP version: 0.52.0


In [5]:
# Import libraries
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)

## 2. Data cleaning

Four distinct statistical datasets spanning 2014 to 2025 are cleaned and standardized to construct a consistent borough-year dataset:

* **UK House Price Index (UK HPI):** Filtered for the six South-West London target boroughs (Croydon, Kingston upon Thames, Merton, Richmond upon Thames, Sutton, Wandsworth). Monthly average property prices are aggregated into annual means (`AveragePrice`).


* **Annual Survey of Hours and Earnings (ASHE):** Extracted workplace-based annual median gross pay for full-time workers (`MedianAnnualPay`) from the London Datastore. If any missing entriesfor a borough in one year, then it was estimated using the values from the years immediately before and after it. For instance, Kingston upon Thames in 2018 earning values was missing but estimated by using the 2017 and 2019 values.


* **ONS Household Saving Ratio:** Processed national annual seasonally adjusted saving ratio figures (`SavingRatio`).


* **Bank of England Policy Base Rate:** Daily base rates are weighted and averaged annually to capture external monetary conditions (`BaseRate`).

In [6]:
# upload data

PROJECT_ROOT = Path.cwd()

RAW_DATA = PROJECT_ROOT / "data_raw"
CLEAN_DATA = PROJECT_ROOT / "data_clean"
OUTPUTS = PROJECT_ROOT / "outputs"
MODELS = PROJECT_ROOT / "models"
FIGURES = OUTPUTS / "figures"


# Make sure folders exist
CLEAN_DATA.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

print("OUTPUTS:", OUTPUTS.relative_to(PROJECT_ROOT))
print("Type:", type(OUTPUTS))

# Original raw files
earnings_raw = pd.read_excel(RAW_DATA / "earnings.xls")
hpi_raw = pd.read_csv(RAW_DATA / "hpi.csv")

bank_raw = pd.read_csv(RAW_DATA / "rate.csv")
saving_raw = pd.read_csv(RAW_DATA / "saving_ratio.csv")

# preview the shape 
print("Earnings:", earnings_raw.shape)
print("HPI:", hpi_raw.shape)
print("Bank Rate:", bank_raw.shape)
print("Saving Ratio:", saving_raw.shape)

OUTPUTS: /Users/sarah/Documents/Codex/2026-08-15/referenced-chatgpt-conversation-this-is-an/FundFirst_Code_Submission_260010319/outputs
Type: <class 'pathlib.PosixPath'>


Earnings: (32, 2)
HPI: (149085, 7)
Bank Rate: (258, 2)
Saving Ratio: (323, 2)


In [7]:
#preview first five rows for the four dataset
def preview(df):
    display(df.head(5))
    
preview(earnings_raw)
preview(hpi_raw)
preview(bank_raw)
preview(saving_raw)

,Name,Annual survey of hours and earnings
0,ShortName,ASHE
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,Theme,Employment and Skills


,Date,Region_Name,Area_Code,Average_Price,Monthly_Change,Annual_Change,Average_Price_SA
0,1968-04-01,Northern Ireland,N92000002,3465,NaN,NaN,NaN
1,1968-04-01,England,E92000001,3218,NaN,NaN,NaN
2,1968-04-01,Wales,W92000004,2732,NaN,NaN,NaN
3,1968-04-01,Scotland,S92000003,2738,NaN,NaN,NaN
4,1968-04-01,London,E12000007,4730,NaN,NaN,NaN


,Date Changed,Rate
0,18 Dec 25,3.75
1,07 Aug 25,4.00
2,08 May 25,4.25
3,06 Feb 25,4.50
4,07 Nov 24,4.75


,Title,Households (S.14): Households' saving ratio (per cent): Current price: £m: SA
0,CDID,DGD8
1,Source dataset ID,UKEA
2,PreUnit,NaN
3,Unit,NaN
4,Release date,30-06-2026


### UK House Price Index
The UK House Price Index (UK HPI) dataset contains monthly average property price records across UK regions and local authorities. FundFirst filters the dataset for the six target South-West London boroughs (Croydon, Kingston upon Thames, Merton, Richmond upon Thames, Sutton, and Wandsworth) covering 2014 to 2025. 

The monthly **Average_Price** (property prices) are cleaned and converted into numbers, then averaged for each year and borough. This gives one annual average property price for each borough-year. The columns and areas outside the six selected boroughs are removed.

In [8]:
hpi = pd.read_csv(RAW_DATA / "hpi.csv")

hpi["Date"] = pd.to_datetime(
    hpi["Date"],
    errors="coerce"
)

hpi["Year"] = hpi["Date"].dt.year

target_boroughs = [
    "Wandsworth",
    "Richmond upon Thames",
    "Kingston upon Thames",
    "Merton",
    "Sutton",
    "Croydon"
]

# Keep only required boroughs and study years
hpi = hpi[
    (hpi["Region_Name"].isin(target_boroughs)) &
    (hpi["Year"].between(2014, 2025))
].copy()

# Keep relevant columns
hpi = hpi[
    ["Date", "Region_Name", "Year", "Average_Price"]
].copy()

# Check missing values and duplicates
print("Missing values:")
print(hpi.isna().sum())

print("\nDuplicate rows:")
print(hpi.duplicated().sum())

Missing values:
Date             0
Region_Name      0
Year             0
Average_Price    0
dtype: int64

Duplicate rows:
0


In [9]:
#Check that every borough-year has 12 monthly observations:
monthly_check = (
    hpi.groupby(["Region_Name", "Year"])
       .size()
       .reset_index(name="Months")
)

display(monthly_check)

print(monthly_check["Months"].value_counts())

,Region_Name,Year,Months
0,Croydon,2014,12
1,Croydon,2015,12
2,Croydon,2016,12
3,Croydon,2017,12
4,Croydon,2018,12
...,...,...,...
67,Wandsworth,2021,12
68,Wandsworth,2022,12
69,Wandsworth,2023,12
70,Wandsworth,2024,12


Months
12    72
Name: count, dtype: int64


In [10]:
hpi_annual = (
    hpi.groupby(
        ["Region_Name", "Year"],
        as_index=False
    )["Average_Price"]
    .mean()
)

hpi_annual = hpi_annual.rename(columns={
    "Region_Name": "Borough",
    "Average_Price": "AveragePrice"
})

hpi_annual["AveragePrice"] = (
    hpi_annual["AveragePrice"].round(2)
)

assert len(hpi_annual) == 72
assert not hpi_annual.duplicated(
    ["Borough", "Year"]
).any()

display(hpi_annual.head(12))

,Borough,Year,AveragePrice
0,Croydon,2014,278641.17
1,Croydon,2015,311325.08
2,Croydon,2016,360784.75
3,Croydon,2017,377724.33
4,Croydon,2018,375426.75
5,Croydon,2019,371015.17
6,Croydon,2020,379844.92
7,Croydon,2021,392932.75
8,Croydon,2022,413625.75
9,Croydon,2023,408826.42


In [11]:
hpi_annual.to_csv(
    CLEAN_DATA / "hpi_annual_clean.csv",
    index=False
)
print("Saved HPI:", hpi_annual.shape)

Saved HPI: (72, 3)


### Earnings
The ASHE earnings spreadsheet provides workplace-based income statistics across multiple demographic sheets. FundFirst extracts annual median gross pay from the FT workers annual Median sheet to derive the **MedianAnnualPay** feature for full-time workers. The records are filtered for the six South-West London boroughs across the 2014 to 2025 timeframe. Data quality steps involve removing the metadata headers, handling missing values (such as #), converting string entries to standard numeric float values, and aligning borough name keys for seamless merging with the HPI dataset at the borough-year level.

In [12]:
# EARNINGS CLEANING
earnings = pd.read_excel(
    RAW_DATA / "earnings.xls",
    sheet_name="FT workers annual Median"
)

years = list(range(2014, 2026))

earnings = earnings[
    earnings["Area"].isin(target_boroughs)
][["Code", "Area"] + years].copy()

display(earnings)

,Code,Area,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
8,E09000008,Croydon,31154.0,30628.0,31479,32109.0,33804,35654,34107,34911.0,38726,38696,43337,40033.0
21,E09000021,Kingston upon Thames,30768.0,28635.0,30729,31308.0,#,32261,35268,35713.0,36203,37469,40719,42346.0
24,E09000024,Merton,27539.0,30040.0,28224,29627.0,31182,31741,33114,34210.0,33980,37158,38736,38471.0
27,E09000027,Richmond upon Thames,31533.0,32660.0,33329,32141.0,34027,35390,35575,35651.0,36888,39523,41536,41037.0
29,E09000029,Sutton,25797.0,27355.0,28318,27945.0,28853,32999,35097,30763.0,35728,35588,39364,39874.0
32,E09000032,Wandsworth,31085.0,31968.0,33405,33137.0,34501,34814,34791,33499.0,36434,39723,39999,43035.0


In [13]:
#convert the year columns to numeric
# Any missing ASHE values (#) were converted to become NaN

for year in years:
    earnings[year] = pd.to_numeric(
        earnings[year],
        errors="coerce"
    )

print("Missing earnings values:")
print(earnings[years].isna().sum())

Missing earnings values:
2014    0
2015    0
2016    0
2017    0
2018    1
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
2025    0
dtype: int64


In [14]:
display(
    earnings[
        earnings[years].isna().any(axis=1)
    ]
)

,Code,Area,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
21,E09000021,Kingston upon Thames,30768.0,28635.0,30729,31308.0,NaN,32261,35268,35713.0,36203,37469,40719,42346.0


In [15]:
earnings[years] = earnings[years].interpolate(
    axis=1,
    limit_area="inside"
)

In [16]:
print("Missing values after interpolation:")
print(earnings[years].isna().sum())

display(
    earnings[
        earnings["Area"] == "Kingston upon Thames"
    ][["Area", 2017, 2018, 2019]]
)

Missing values after interpolation:
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
2025    0
dtype: int64


,Area,2017,2018,2019
21,Kingston upon Thames,31308.0,31784.5,32261.0


In [17]:
# Final missing-value check after the single interpolation step
print("Missing earnings values after interpolation:")
print(earnings[years].isna().sum())

display(
    earnings[
        earnings[years].isna().any(axis=1)
    ]
)


Missing earnings values after interpolation:
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
2025    0
dtype: int64


,Code,Area,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025


In [18]:
earnings_annual = earnings.melt(
    id_vars=["Code", "Area"],
    value_vars=years,
    var_name="Year",
    value_name="MedianAnnualPay"
)

earnings_annual = earnings_annual.rename(
    columns={"Area": "Borough"}
)

earnings_annual["Year"] = earnings_annual["Year"].astype(int)

earnings_annual = earnings_annual[
    ["Borough", "Year", "MedianAnnualPay"]
].sort_values(
    ["Borough", "Year"]
).reset_index(drop=True)

display(earnings_annual.head(12))

print("Shape:", earnings_annual.shape)
print("Missing:", earnings_annual.isna().sum())

,Borough,Year,MedianAnnualPay
0,Croydon,2014,31154.0
1,Croydon,2015,30628.0
2,Croydon,2016,31479.0
3,Croydon,2017,32109.0
4,Croydon,2018,33804.0
5,Croydon,2019,35654.0
6,Croydon,2020,34107.0
7,Croydon,2021,34911.0
8,Croydon,2022,38726.0
9,Croydon,2023,38696.0


Shape: (72, 3)
Missing: Borough            0
Year               0
MedianAnnualPay    0
dtype: int64


In [19]:
assert len(earnings_annual) == 72
assert not earnings_annual.duplicated(
    ["Borough", "Year"]
).any()

earnings_annual.to_csv(
    CLEAN_DATA / "earnings_annual_clean.csv",
    index=False
)

### Household saving ratio

The ONS Household Saving Ratio file contains metadata followed by annual and quarterly observations. FundFirst uses the published annual,
seasonally adjusted values for 2014 to 2025. 
**SavingRatio** is a national time-varying feature and is not used to generate the feasibility labels.

In [20]:
# SAVING RATIO CLEANING

saving = pd.read_csv(
    RAW_DATA / "saving_ratio.csv",
    skiprows=8,
    header=None,
    names=["Period", "SavingRatio"]
)

display(saving.head())

,Period,SavingRatio
0,1963,4.9
1,1964,5.9
2,1965,6.2
3,1966,6.5
4,1967,5.8


In [21]:
# Convert period to text
saving["Period"] = saving["Period"].astype(str).str.strip()

# Keep only four-digit annual observations
saving = saving[
    saving["Period"].str.fullmatch(r"\d{4}")
].copy()

# Convert types
saving["Year"] = pd.to_numeric(
    saving["Period"],
    errors="coerce"
)

saving["SavingRatio"] = pd.to_numeric(
    saving["SavingRatio"],
    errors="coerce"
)

# Keep FundFirst study period
saving_annual = saving[
    saving["Year"].between(2014, 2025)
][["Year", "SavingRatio"]].copy()

saving_annual["Year"] = saving_annual["Year"].astype(int)

display(saving_annual)

,Year,SavingRatio
51,2014,7.3
52,2015,9.7
53,2016,6.3
54,2017,5.2
55,2018,5.4
56,2019,5.8
57,2020,16.6
58,2021,12.7
59,2022,5.4
60,2023,6.2


In [22]:
print("Shape:", saving_annual.shape)

print("\nMissing values:")
print(saving_annual.isna().sum())

print("\nDuplicate years:")
print(saving_annual.duplicated(["Year"]).sum())

assert len(saving_annual) == 12
assert saving_annual["Year"].is_unique
assert saving_annual["SavingRatio"].notna().all()

Shape: (12, 2)

Missing values:
Year           0
SavingRatio    0
dtype: int64

Duplicate years:
0


In [23]:
saving_annual.to_csv(
    CLEAN_DATA / "saving_ratio_annual_clean.csv",
    index=False
)

print("Saving Ratio complete.")

Saving Ratio complete.


### Bank Rate

The Bank of England file records dates on which Bank Rate changed rather than regular monthly observations. Each rate is therefore carried forward until the next change date and converted into a daily series. A
time-weighted annual mean is then calculated for 2014 to 2025.

In [24]:
# BANK RATE CLEANING

bank = bank_raw.copy()

bank = bank.rename(columns={
    "Date Changed": "Date",
    "Rate": "BaseRate"
})

display(bank.head())

,Date,BaseRate
0,18 Dec 25,3.75
1,07 Aug 25,4.00
2,08 May 25,4.25
3,06 Feb 25,4.50
4,07 Nov 24,4.75


In [25]:
bank["Date"] = pd.to_datetime(
    bank["Date"],
    format="%d %b %y",
    errors="coerce"
)

bank["BaseRate"] = pd.to_numeric(
    bank["BaseRate"],
    errors="coerce"
)

# Remove invalid observations if any
bank = bank.dropna(
    subset=["Date", "BaseRate"]
)

# Oldest to newest
bank = bank.sort_values("Date")

# Ensure one observation per change date
bank = bank.drop_duplicates(
    subset=["Date"],
    keep="last"
).reset_index(drop=True)

display(bank.head())
display(bank.tail())

,Date,BaseRate
0,1975-01-20,11.25
1,1975-01-27,11.00
2,1975-02-10,10.75
3,1975-02-17,10.50
4,1975-03-10,10.25


,Date,BaseRate
253,2024-11-07,4.75
254,2025-02-06,4.50
255,2025-05-08,4.25
256,2025-08-07,4.00
257,2025-12-18,3.75


In [26]:
print("Earliest:", bank["Date"].min())
print("Latest:", bank["Date"].max())

print("\nMissing values:")
print(bank.isna().sum())

Earliest: 1975-01-20 00:00:00
Latest: 2025-12-18 00:00:00

Missing values:
Date        0
BaseRate    0
dtype: int64


In [27]:
bank_calendar = pd.DataFrame({
    "Date": pd.date_range(
        start="2014-01-01",
        end="2025-12-31",
        freq="D"
    )
})

In [28]:
bank_daily = pd.merge_asof(
    bank_calendar.sort_values("Date"),
    bank[["Date", "BaseRate"]].sort_values("Date"),
    on="Date",
    direction="backward"
)

bank_daily["Year"] = bank_daily["Date"].dt.year

display(bank_daily.head())

,Date,BaseRate,Year
0,2014-01-01,0.5,2014
1,2014-01-02,0.5,2014
2,2014-01-03,0.5,2014
3,2014-01-04,0.5,2014
4,2014-01-05,0.5,2014


In [29]:
print("Missing daily rates:")
print(bank_daily["BaseRate"].isna().sum())

assert bank_daily["BaseRate"].notna().all()

Missing daily rates:
0


In [30]:
bank_annual = (
    bank_daily
    .groupby("Year", as_index=False)
    .agg(
        BaseRate=("BaseRate", "mean"),
        NumberOfDays=("BaseRate", "size")
    )
)

bank_annual["BaseRate"] = (
    bank_annual["BaseRate"].round(4)
)

display(bank_annual)

,Year,BaseRate,NumberOfDays
0,2014,0.5000,365
1,2015,0.5000,365
2,2016,0.3975,366
3,2017,0.2911,365
4,2018,0.6041,365
5,2019,0.7500,365
6,2020,0.2276,366
7,2021,0.1066,365
8,2022,1.4658,365
9,2023,4.6795,365


In [31]:
print("Shape:", bank_annual.shape)

print("\nMissing values:")
print(bank_annual.isna().sum())

print("\nDuplicate years:")
print(bank_annual.duplicated(["Year"]).sum())

assert len(bank_annual) == 12
assert bank_annual["Year"].is_unique
assert bank_annual["BaseRate"].notna().all()

Shape: (12, 3)

Missing values:
Year            0
BaseRate        0
NumberOfDays    0
dtype: int64

Duplicate years:
0


In [32]:
bank_annual_clean = bank_annual[
    ["Year", "BaseRate"]
].copy()

bank_annual_clean.to_csv(
    CLEAN_DATA / "base_rate_annual_clean.csv",
    index=False
)

display(bank_annual_clean)

print("Bank Rate complete.")

,Year,BaseRate
0,2014,0.5000
1,2015,0.5000
2,2016,0.3975
3,2017,0.2911
4,2018,0.6041
5,2019,0.7500
6,2020,0.2276
7,2021,0.1066
8,2022,1.4658
9,2023,4.6795


Bank Rate complete.


In [33]:
print("HPI:", hpi_annual.shape)
print("Earnings:", earnings_annual.shape)
print("Saving Ratio:", saving_annual.shape)
print("Bank Rate:", bank_annual_clean.shape)

HPI: (72, 3)
Earnings: (72, 3)
Saving Ratio: (12, 2)
Bank Rate: (12, 2)


## 3. Dataset assembly

All four cleaned datasets are inner-joined on **Year** and **Borough**. The four cleaned datasets are merged at borough-year level. AveragePrice and MedianAnnualPay vary by borough and year. SavingRatio and BaseRate are national time-varying features and therefore have the same value across all six boroughs within each year.

In [34]:
fundfirst = (
    hpi_annual
    .merge(
        earnings_annual,
        on=["Borough", "Year"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        saving_annual,
        on="Year",
        how="left",
        validate="many_to_one"
    )
    .merge(
        bank_annual_clean,
        on="Year",
        how="left",
        validate="many_to_one"
    )
)

display(fundfirst.head(12))

,Borough,Year,AveragePrice,MedianAnnualPay,SavingRatio,BaseRate
0,Croydon,2014,278641.17,31154.0,7.3,0.5000
1,Croydon,2015,311325.08,30628.0,9.7,0.5000
2,Croydon,2016,360784.75,31479.0,6.3,0.3975
3,Croydon,2017,377724.33,32109.0,5.2,0.2911
4,Croydon,2018,375426.75,33804.0,5.4,0.6041
5,Croydon,2019,371015.17,35654.0,5.8,0.7500
6,Croydon,2020,379844.92,34107.0,16.6,0.2276
7,Croydon,2021,392932.75,34911.0,12.7,0.1066
8,Croydon,2022,413625.75,38726.0,5.4,1.4658
9,Croydon,2023,408826.42,38696.0,6.2,4.6795


## 4. Dataset validation
The merged dataset is validated against strict completeness criteria:  

- Total Rows: Exactly 72 observations (6 South-West London boroughs × 12 years: 2014–2025).  

- Schema: 'Borough', 'Year', 'AveragePrice', 'MedianAnnualPay', 'SavingRatio', 'BaseRate'.  

- Data Quality Check: Verified zero unhandled missing or null values across all 72 rows.

In [35]:
print("Shape:", fundfirst.shape)

print("\nMissing values:")
print(fundfirst.isna().sum())

print("\nDuplicate borough-years:")
print(
    fundfirst.duplicated(
        ["Borough", "Year"]
    ).sum()
)

Shape: (72, 6)

Missing values:
Borough            0
Year               0
AveragePrice       0
MedianAnnualPay    0
SavingRatio        0
BaseRate           0
dtype: int64

Duplicate borough-years:
0


In [36]:
assert len(fundfirst) == 72

assert not fundfirst.duplicated(
    ["Borough", "Year"]
).any()

assert fundfirst.isna().sum().sum() == 0

In [37]:
assert (
    fundfirst.groupby("Year")["SavingRatio"]
    .nunique()
    .max()
    == 1
)

assert (
    fundfirst.groupby("Year")["BaseRate"]
    .nunique()
    .max()
    == 1
)

print("Merged dataset validation passed.")

Merged dataset validation passed.


In [38]:
fundfirst.to_csv(
    CLEAN_DATA / "fundfirst_features_clean.csv",
    index=False
)

## 5. Time–Savings–Market rule 

The deterministic TSM rule generates the target feasibility labels. A 10% deposit target and a calibrated 20% active saving rate are used. The observed ONS SavingRatio is not used in the label-generation formula. The TSM deterministic rule computes deposit timeline feasibility step-by-step:

1. **Deposit Target:** $\text{AveragePrice} \times 0.10$

2. **Monthly Income:** $\frac{\text{MedianAnnualPay}}{12}$

3. **Monthly Savings:** $\text{Monthly Income} \times 0.20$ (assuming a 20% savings rate for prospective deposit buyers)


4. **Months to Save:** $\frac{\text{Deposit Target}}{\text{Monthly Savings}}$

In [39]:
DEPOSIT_RATE = 0.10
ACTIVE_SAVING_RATE = 0.20

In [40]:
# TSM Computation 
fundfirst["DepositTarget"] = (
    fundfirst["AveragePrice"] *
    DEPOSIT_RATE
)

fundfirst["MonthlyIncome"] = (
    fundfirst["MedianAnnualPay"] / 12
)

fundfirst["MonthlySavings"] = (
    fundfirst["MonthlyIncome"] *
    ACTIVE_SAVING_RATE
)

fundfirst["MonthsToSave"] = (
    fundfirst["DepositTarget"] /
    fundfirst["MonthlySavings"]
)

In [41]:
display(
    fundfirst[
        [
            "Borough",
            "Year",
            "AveragePrice",
            "MedianAnnualPay",
            "DepositTarget",
            "MonthlySavings",
            "MonthsToSave"
        ]
    ].head(12)
)

,Borough,Year,AveragePrice,MedianAnnualPay,DepositTarget,MonthlySavings,MonthsToSave
0,Croydon,2014,278641.17,31154.0,27864.117,519.233333,53.663960
1,Croydon,2015,311325.08,30628.0,31132.508,510.466667,60.988327
2,Croydon,2016,360784.75,31479.0,36078.475,524.650000,68.766749
3,Croydon,2017,377724.33,32109.0,37772.433,535.150000,70.582889
4,Croydon,2018,375426.75,33804.0,37542.675,563.400000,66.635916
5,Croydon,2019,371015.17,35654.0,37101.517,594.233333,62.435940
6,Croydon,2020,379844.92,34107.0,37984.492,568.450000,66.821166
7,Croydon,2021,392932.75,34911.0,39293.275,581.850000,67.531623
8,Croydon,2022,413625.75,38726.0,41362.575,645.433333,64.084969
9,Croydon,2023,408826.42,38696.0,40882.642,644.933333,63.390493


### Initial feasibility tiers

In [42]:
# Define the initial feasibility tiers
def assign_feasibility_tier(months):
    if months <= 48:
        return "Achievable"
    elif months <= 108:
        return "Stretch"
    else:
        return "Unfeasible"

In [43]:
fundfirst["FeasibilityTier"] = (
    fundfirst["MonthsToSave"]
    .apply(assign_feasibility_tier)
)

In [44]:
display(
    fundfirst[
        [
            "Borough",
            "Year",
            "MonthsToSave",
            "FeasibilityTier"
        ]
    ].head(20)
)

,Borough,Year,MonthsToSave,FeasibilityTier
0,Croydon,2014,53.663960,Stretch
1,Croydon,2015,60.988327,Stretch
2,Croydon,2016,68.766749,Stretch
3,Croydon,2017,70.582889,Stretch
4,Croydon,2018,66.635916,Stretch
5,Croydon,2019,62.435940,Stretch
6,Croydon,2020,66.821166,Stretch
7,Croydon,2021,67.531623,Stretch
8,Croydon,2022,64.084969,Stretch
9,Croydon,2023,63.390493,Stretch


### Initial label generation

Each borough-year profile is mapped into one of three feasibility tiers based on `MonthsToSave`:

* **Achievable:** $\le 48 \text{ months}$ ($\le 4 \text{ years}$)


* **Stretch:** $49 \text{ to } 108 \text{ months}$ ($4 \text{ to } 9 \text{ years}$)


* **Unfeasible:** $> 108 \text{ months}$ ($> 9 \text{ years}$)

In [45]:
# Check label distribution
print("Label counts:")
print(
    fundfirst["FeasibilityTier"]
    .value_counts()
)

print("\nMonths to save summary:")
print(
    fundfirst["MonthsToSave"]
    .describe()
    .round(1)
)

print("\nClasses present:")
print(
    fundfirst["FeasibilityTier"]
    .unique()
)

Label counts:
FeasibilityTier
Stretch       48
Unfeasible    24
Name: count, dtype: int64

Months to save summary:


count     72.0
mean      95.0
std       22.5
min       53.7
25%       73.2
50%       97.8
75%      116.5
max      131.7
Name: MonthsToSave, dtype: float64

Classes present:
['Stretch' 'Unfeasible']


In [46]:
CLASS_ORDER = [
    "Achievable",
    "Stretch",
    "Unfeasible"
]

missing_classes = (
    set(CLASS_ORDER)
    - set(fundfirst["FeasibilityTier"].unique())
)

print("Missing classes:", missing_classes)

Missing classes: {'Achievable'}


In [47]:
display(
    fundfirst[
        [
            "Borough",
            "Year",
            "AveragePrice",
            "MedianAnnualPay",
            "MonthsToSave",
            "FeasibilityTier"
        ]
    ]
    .sort_values(["Borough", "Year"])
    .head(20)
)

,Borough,Year,AveragePrice,MedianAnnualPay,MonthsToSave,FeasibilityTier
0,Croydon,2014,278641.17,31154.0,53.663960,Stretch
1,Croydon,2015,311325.08,30628.0,60.988327,Stretch
2,Croydon,2016,360784.75,31479.0,68.766749,Stretch
3,Croydon,2017,377724.33,32109.0,70.582889,Stretch
4,Croydon,2018,375426.75,33804.0,66.635916,Stretch
5,Croydon,2019,371015.17,35654.0,62.435940,Stretch
6,Croydon,2020,379844.92,34107.0,66.821166,Stretch
7,Croydon,2021,392932.75,34911.0,67.531623,Stretch
8,Croydon,2022,413625.75,38726.0,64.084969,Stretch
9,Croydon,2023,408826.42,38696.0,63.390493,Stretch


## 6. Threshold sensitivity analysis

The initial Achievable threshold of 48 months produced no Achievable observations, with the minimum estimated saving period equal to 53.7 months. A sensitivity analysis was therefore conducted to assess the effect of alternative lower thresholds while retaining the 108-month
boundary for the Unfeasible class. 

To reduce test-set influence on label calibration, candidate thresholds were assessed using the 2014–2022 development period only. The final 2024–2025 period was not used to select the threshold.

### Candidate-threshold class balance

In [48]:
candidate_thresholds = [48, 54, 60, 66, 72]

STRETCH_MAX = 108

# Use earlier years only for threshold calibration
calibration_df = fundfirst[
    fundfirst["Year"].between(2014, 2022)
].copy()

sensitivity_results = []

for achievable_max in candidate_thresholds:

    temporary_labels = np.select(
        [
            calibration_df["MonthsToSave"] <= achievable_max,
            calibration_df["MonthsToSave"] <= STRETCH_MAX
        ],
        [
            "Achievable",
            "Stretch"
        ],
        default="Unfeasible"
    )

    counts = pd.Series(temporary_labels).value_counts()

    sensitivity_results.append({
        "AchievableMaxMonths": achievable_max,
        "Achievable": counts.get("Achievable", 0),
        "Stretch": counts.get("Stretch", 0),
        "Unfeasible": counts.get("Unfeasible", 0)
    })

sensitivity_table = pd.DataFrame(
    sensitivity_results
)

display(sensitivity_table)

,AchievableMaxMonths,Achievable,Stretch,Unfeasible
0,48,0,34,20
1,54,1,33,20
2,60,1,33,20
3,66,5,29,20
4,72,13,21,20


### Candidate-boundary diagnostics

In [49]:
# Summary statistics for MonthsToSave
print(
    calibration_df["MonthsToSave"]
    .describe()
    .round(1)
)

# Show the 25 lowest MonthsToSave observations
display(
    calibration_df[
        [
            "Borough",
            "Year",
            "AveragePrice",
            "MedianAnnualPay",
            "MonthsToSave"
        ]
    ]
    .sort_values("MonthsToSave")
    .head(25)
)

count     54.0
mean      97.0
std       22.9
min       53.7
25%       74.9
50%       99.1
75%      120.2
max      131.7
Name: MonthsToSave, dtype: float64


,Borough,Year,AveragePrice,MedianAnnualPay,MonthsToSave
0,Croydon,2014,278641.17,31154.0,53.663960
1,Croydon,2015,311325.08,30628.0,60.988327
5,Croydon,2019,371015.17,35654.0,62.435940
8,Croydon,2022,413625.75,38726.0,64.084969
54,Sutton,2020,382900.00,35097.0,65.458586
4,Croydon,2018,375426.75,33804.0,66.635916
6,Croydon,2020,379844.92,34107.0,66.821166
7,Croydon,2021,392932.75,34911.0,67.531623
53,Sutton,2019,375669.50,32999.0,68.305615
48,Sutton,2014,294322.67,25797.0,68.455092


### Borough coverage by threshold

In [50]:
# BOROUGH COVERAGE SENSITIVITY CHECK

for threshold in candidate_thresholds:

    temp = calibration_df.copy()

    temp["TemporaryTier"] = np.select(
        [
            temp["MonthsToSave"] <= threshold,
            temp["MonthsToSave"] <= STRETCH_MAX
        ],
        [
            "Achievable",
            "Stretch"
        ],
        default="Unfeasible"
    )

    achievable_rows = temp[
        temp["TemporaryTier"] == "Achievable"
    ]

    print(f"\nThreshold <= {threshold} months")

    print(
        "Achievable rows:",
        len(achievable_rows)
    )

    print(
        "Boroughs represented:",
        achievable_rows["Borough"].nunique()
    )

    print(
        achievable_rows["Borough"].value_counts()
    )


Threshold <= 48 months
Achievable rows: 0
Boroughs represented: 0
Series([], Name: count, dtype: int64)

Threshold <= 54 months
Achievable rows: 1
Boroughs represented: 1
Borough
Croydon    1
Name: count, dtype: int64

Threshold <= 60 months
Achievable rows: 1
Boroughs represented: 1
Borough
Croydon    1
Name: count, dtype: int64

Threshold <= 66 months
Achievable rows: 5
Boroughs represented: 2
Borough
Croydon    4
Sutton     1
Name: count, dtype: int64

Threshold <= 72 months
Achievable rows: 13
Boroughs represented: 2
Borough
Croydon    9
Sutton     4
Name: count, dtype: int64


### Threshold Calibration Decision

The initial 48-month Achievable threshold produced no Achievable observations within the 2014–2022 calibration period. Sensitivity
analysis was therefore conducted at 48, 54, 60, 66 and 72 months while retaining the original 108-month upper boundary.

Thresholds of 54 and 60 months produced only one Achievable observation, while 66 months produced five. A 72-month boundary produced 13
Achievable, 21 Stretch and 20 Unfeasible observations in the calibration period. It was therefore selected as the project-specific operational boundary because it retained the intended three-class structure while providing sufficient observations for supervised modelling.

Note: this threshold is a modelling calibration for this South West London dataset and should not be interpreted as a universal definition of housing affordability.

In [51]:
# FINAL TSM THRESHOLDS

ACHIEVABLE_MAX = 72
STRETCH_MAX = 108

print("Final calibrated thresholds:")
print(f"Achievable: <= {ACHIEVABLE_MAX} months")
print(
    f"Stretch: {ACHIEVABLE_MAX + 1}–"
    f"{STRETCH_MAX} months"
)
print(f"Unfeasible: > {STRETCH_MAX} months")

Final calibrated thresholds:
Achievable: <= 72 months
Stretch: 73–108 months
Unfeasible: > 108 months


In [52]:
def assign_feasibility_tier(months):

    if months <= ACHIEVABLE_MAX:
        return "Achievable"

    elif months <= STRETCH_MAX:
        return "Stretch"

    else:
        return "Unfeasible"


fundfirst["FeasibilityTier"] = (
    fundfirst["MonthsToSave"]
    .apply(assign_feasibility_tier)
)

In [53]:
print("Overall label counts:")

print(
    fundfirst["FeasibilityTier"]
    .value_counts()
)

print("\nClasses present:")

print(
    fundfirst["FeasibilityTier"]
    .unique()
)

Overall label counts:
FeasibilityTier
Stretch       30
Unfeasible    24
Achievable    18
Name: count, dtype: int64

Classes present:
['Achievable' 'Stretch' 'Unfeasible']


### Rule-based baseline sanity check

In [54]:
# BASELINE SELF-AGREEMENT

def rule_based_baseline(months):

    if months <= ACHIEVABLE_MAX:
        return "Achievable"

    elif months <= STRETCH_MAX:
        return "Stretch"

    else:
        return "Unfeasible"


fundfirst["BaselinePrediction"] = (
    fundfirst["MonthsToSave"]
    .apply(rule_based_baseline)
)

agreement = (
    fundfirst["BaselinePrediction"]
    ==
    fundfirst["FeasibilityTier"]
).mean()

print(
    f"Baseline self-agreement: {agreement:.1%}"
)

if agreement == 1.0:
    print("Baseline check: SUPPORTED")
else:
    print("Baseline check: CHECK IMPLEMENTATION")

Baseline self-agreement: 100.0%
Baseline check: SUPPORTED


In [55]:
class_counts = (
    fundfirst["FeasibilityTier"]
    .value_counts()
)

print(class_counts)

FeasibilityTier
Stretch       30
Unfeasible    24
Achievable    18
Name: count, dtype: int64


In [56]:
# Check the training dataset for 2014-2022 before modelling - sanity check 
calibration_labels = fundfirst[
    fundfirst["Year"].between(2014, 2022)
]

print("2014–2022 class distribution:")

print(
    calibration_labels[
        "FeasibilityTier"
    ].value_counts()
)

2014–2022 class distribution:
FeasibilityTier
Stretch       21
Unfeasible    20
Achievable    13
Name: count, dtype: int64


### Final class validation

In [57]:
CLASS_ORDER = [
    "Achievable",
    "Stretch",
    "Unfeasible"
]

missing_train_classes = (
    set(CLASS_ORDER)
    -
    set(
        calibration_labels[
            "FeasibilityTier"
        ].unique()
    )
)

print(
    "Classes missing from training:",
    missing_train_classes
)

Classes missing from training: set()


In [58]:
fundfirst.to_csv(
    CLEAN_DATA /
    "fundfirst_labelled_dataset.csv",
    index=False
)

print(
    "Saved final labelled dataset:",
    fundfirst.shape
)

Saved final labelled dataset: (72, 12)


## 7. Chronological train, validation and test split

To evaluate model generalisation without data leakage across time, data is split chronologically:

* **Training Set (2014–2022):** 54 observations (75%)


* **Validation Set (2023):** 6 observations (8.3%)


* **Test Set (2024–2025):** 12 observations (16.7%)


In [59]:
# FEATURES AND TARGET
FEATURES = [
    "AveragePrice",
    "MedianAnnualPay",
    "SavingRatio",
    "BaseRate"
]

TARGET = "FeasibilityTier"

# TRAIN / VALIDATION / TEST SPLIT

train_df = fundfirst[
    fundfirst["Year"].between(2014, 2022)
].copy()

validation_df = fundfirst[
    fundfirst["Year"] == 2023
].copy()

test_df = fundfirst[
    fundfirst["Year"].between(2024, 2025)
].copy()

print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

print("\nTraining class distribution:")
print(train_df[TARGET].value_counts())

print("\nValidation class distribution:")
print(validation_df[TARGET].value_counts())

Training rows: 54
Validation rows: 6
Test rows: 12

Training class distribution:
FeasibilityTier
Stretch       21
Unfeasible    20
Achievable    13
Name: count, dtype: int64

Validation class distribution:
FeasibilityTier
Stretch       4
Achievable    1
Unfeasible    1
Name: count, dtype: int64


In [60]:
display(
    train_df[
        [
            "Borough",
            "Year",
            "AveragePrice",
            "MedianAnnualPay",
            "SavingRatio",
            "BaseRate",
            "FeasibilityTier"
        ]
    ].head(10)
)

,Borough,Year,AveragePrice,MedianAnnualPay,SavingRatio,BaseRate,FeasibilityTier
0,Croydon,2014,278641.17,31154.0,7.3,0.5000,Achievable
1,Croydon,2015,311325.08,30628.0,9.7,0.5000,Achievable
2,Croydon,2016,360784.75,31479.0,6.3,0.3975,Achievable
3,Croydon,2017,377724.33,32109.0,5.2,0.2911,Achievable
4,Croydon,2018,375426.75,33804.0,5.4,0.6041,Achievable
5,Croydon,2019,371015.17,35654.0,5.8,0.7500,Achievable
6,Croydon,2020,379844.92,34107.0,16.6,0.2276,Achievable
7,Croydon,2021,392932.75,34911.0,12.7,0.1066,Achievable
8,Croydon,2022,413625.75,38726.0,5.4,1.4658,Achievable
12,Kingston upon Thames,2014,427740.00,30768.0,7.3,0.5000,Stretch


In [61]:
print("TRAINING PREVIEW")
display(train_df.head(5))

print("VALIDATION PREVIEW")
display(validation_df.head(5))

print("TEST PREVIEW")
display(test_df.head(5))

TRAINING PREVIEW


,Borough,Year,AveragePrice,MedianAnnualPay,SavingRatio,BaseRate,DepositTarget,MonthlyIncome,MonthlySavings,MonthsToSave,FeasibilityTier,BaselinePrediction
0,Croydon,2014,278641.17,31154.0,7.3,0.5000,27864.117,2596.166667,519.233333,53.663960,Achievable,Achievable
1,Croydon,2015,311325.08,30628.0,9.7,0.5000,31132.508,2552.333333,510.466667,60.988327,Achievable,Achievable
2,Croydon,2016,360784.75,31479.0,6.3,0.3975,36078.475,2623.250000,524.650000,68.766749,Achievable,Achievable
3,Croydon,2017,377724.33,32109.0,5.2,0.2911,37772.433,2675.750000,535.150000,70.582889,Achievable,Achievable
4,Croydon,2018,375426.75,33804.0,5.4,0.6041,37542.675,2817.000000,563.400000,66.635916,Achievable,Achievable


VALIDATION PREVIEW


,Borough,Year,AveragePrice,MedianAnnualPay,SavingRatio,BaseRate,DepositTarget,MonthlyIncome,MonthlySavings,MonthsToSave,FeasibilityTier,BaselinePrediction
9,Croydon,2023,408826.42,38696.0,6.2,4.6795,40882.642,3224.666667,644.933333,63.390493,Achievable,Achievable
21,Kingston upon Thames,2023,584057.42,37469.0,6.2,4.6795,58405.742,3122.416667,624.483333,93.526502,Stretch,Stretch
33,Merton,2023,605562.42,37158.0,6.2,4.6795,60556.242,3096.500000,619.300000,97.781757,Stretch,Stretch
45,Richmond upon Thames,2023,790455.92,39523.0,6.2,4.6795,79045.592,3293.583333,658.716667,119.999381,Unfeasible,Unfeasible
57,Sutton,2023,439402.42,35588.0,6.2,4.6795,43940.242,2965.666667,593.133333,74.081559,Stretch,Stretch


TEST PREVIEW


,Borough,Year,AveragePrice,MedianAnnualPay,SavingRatio,BaseRate,DepositTarget,MonthlyIncome,MonthlySavings,MonthsToSave,FeasibilityTier,BaselinePrediction
10,Croydon,2024,398770.42,43337.0,9.8,5.1079,39877.042,3611.416667,722.283333,55.209694,Achievable,Achievable
11,Croydon,2025,401758.67,40033.0,9.6,4.2514,40175.867,3336.083333,667.216667,60.214124,Achievable,Achievable
22,Kingston upon Thames,2024,571939.33,40719.0,9.8,5.1079,57193.933,3393.250000,678.650000,84.276038,Stretch,Stretch
23,Kingston upon Thames,2025,578451.67,42346.0,9.6,4.2514,57845.167,3528.833333,705.766667,81.960752,Stretch,Stretch
34,Merton,2024,602899.33,38736.0,9.8,5.1079,60289.933,3228.000000,645.600000,93.385894,Stretch,Stretch


## 8. Model training

**RQ1.** To what extent can a multi-class classification model reproduce the feasibility tiers (Achievable, Stretch, and Unfeasible) for prospective first-time buyers in South-West London

### Model Configuration

Logistic Regression and Random Forest were trained using fixed hyperparameters to limit overfitting and unnecessary tuning on the small
dataset. Logistic Regression used a maximum of 2,000 optimisation iterations to provide sufficient opportunity for convergence after
feature standardisation.

Random Forest used 300 decision trees to provide a stable ensemble while keeping computational complexity proportionate to the small training dataset. Balanced class weights were used for both models because the three feasibility classes were not equally represented.

A fixed random state of 42 was used to support reproducibility. Extensive hyperparameter optimisation was not performed because the dataset contained only 54 training observations.

### Logistic Regression

* Serves as the transparent linear baseline.


* Features (`AveragePrice`, `MedianAnnualPay`, `SavingRatio`, `BaseRate`) are standardised using `StandardScaler` fitted on the training split.


* Evaluated on validation and holdout test sets.


In [62]:
RANDOM_STATE = 42

# Model inputs and target
FEATURES = [
    "AveragePrice",
    "MedianAnnualPay",
    "SavingRatio",
    "BaseRate"
]

TARGET = "FeasibilityTier"

# Chronological split
train_df = fundfirst[
    fundfirst["Year"].between(2014, 2022)
].copy()

validation_df = fundfirst[
    fundfirst["Year"] == 2023
].copy()

test_df = fundfirst[
    fundfirst["Year"].between(2024, 2025)
].copy()

# Create X and y
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_validation = validation_df[FEATURES]
y_validation = validation_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

# Check before training
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

# Logistic Regression pipeline
logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

# Train
logistic_model.fit(
    X_train,
    y_train
)

print("\nLogistic Regression trained")

X_train shape: (54, 4)
y_train shape: (54,)

Training class distribution:
FeasibilityTier
Stretch       21
Unfeasible    20
Achievable    13
Name: count, dtype: int64



Logistic Regression trained


### Random Forest

* Provides a non-linear comparator to Logistic Regression.
* Uses **300 trees**, balanced class weights, a fixed random state of 42, and all available CPU cores through `n_jobs=-1`.
* No extensive hyperparameter search is performed because the training dataset contains only 54 observations.
* Evaluated first on the 2023 validation checkpoint and later on the untouched 2024–2025 held-out test period using frozen settings.

In [63]:
RANDOM_STATE = 42

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


## 9. Performance evaluation

Both models are evaluated using the same classification metrics:

* overall accuracy
* macro F1
* per-class precision and recall through the classification report
* confusion matrices

The 2023 observations are used as a validation checkpoint. After model settings are frozen, both models are refitted on 2014–2023 and evaluated once on the untouched 2024–2025 held-out period.



In [64]:
# MODEL EVALUATION FUNCTION

CLASS_ORDER = [
    "Achievable",
    "Stretch",
    "Unfeasible"
]

def evaluate_model(model, X, y, model_name):

    predictions = model.predict(X)

    accuracy = accuracy_score(
        y,
        predictions
    )

    macro_f1 = f1_score(
        y,
        predictions,
        average="macro",
        zero_division=0
    )

    print(f"\n{model_name}")
    print("-" * 40)

    print(
        f"Accuracy: {accuracy:.3f}"
    )

    print(
        f"Macro F1: {macro_f1:.3f}"
    )

    print("\nClassification Report:")

    print(
        classification_report(
            y,
            predictions,
            labels=CLASS_ORDER,
            zero_division=0
        )
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "MacroF1": macro_f1
    }, predictions

In [65]:
# Evaluate both models on the 2023 validation checkpoint
lr_validation_results, lr_validation_predictions = evaluate_model(
    logistic_model,
    X_validation,
    y_validation,
    "Logistic Regression"
)

rf_validation_results, rf_validation_predictions = evaluate_model(
    random_forest_model,
    X_validation,
    y_validation,
    "Random Forest"
)



Logistic Regression
----------------------------------------
Accuracy: 0.667
Macro F1: 0.267

Classification Report:
              precision    recall  f1-score   support

  Achievable       0.00      0.00      0.00         1
     Stretch       0.67      1.00      0.80         4
  Unfeasible       0.00      0.00      0.00         1

    accuracy                           0.67         6
   macro avg       0.22      0.33      0.27         6
weighted avg       0.44      0.67      0.53         6


Random Forest
----------------------------------------
Accuracy: 0.833
Macro F1: 0.841

Classification Report:
              precision    recall  f1-score   support

  Achievable       1.00      1.00      1.00         1
     Stretch       1.00      0.75      0.86         4
  Unfeasible       0.50      1.00      0.67         1

    accuracy                           0.83         6
   macro avg       0.83      0.92      0.84         6
weighted avg       0.92      0.83      0.85         6



### 2023 validation results

In [66]:
# Logistic Regression: 2023 validation confusion matrix
lr_val_display = ConfusionMatrixDisplay.from_predictions(
    y_validation,
    lr_validation_predictions,
    labels=CLASS_ORDER,
    xticks_rotation=30,
    cmap="Blues"
)

lr_val_display.ax_.set_title(
    "Logistic Regression - 2023 Validation",
    fontsize=13,
    fontweight="bold"
)
lr_val_display.ax_.set_xlabel("Predicted Class")
lr_val_display.ax_.set_ylabel("Actual Class")

plt.tight_layout()
lr_val_display.figure_.savefig(
    FIGURES / "lr_validation_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/1492429859.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [67]:
# Random Forest: 2023 validation confusion matrix
rf_val_display = ConfusionMatrixDisplay.from_predictions(
    y_validation,
    rf_validation_predictions,
    labels=CLASS_ORDER,
    xticks_rotation=30,
    cmap="Blues"
)

rf_val_display.ax_.set_title(
    "Random Forest - 2023 Validation",
    fontsize=13,
    fontweight="bold"
)
rf_val_display.ax_.set_xlabel("Predicted Class")
rf_val_display.ax_.set_ylabel("Actual Class")

plt.tight_layout()
rf_val_display.figure_.savefig(
    FIGURES / "rf_validation_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/2653965515.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [68]:
# 2023 validation predictions by borough
validation_predictions = validation_df[
    ["Borough", "Year", "FeasibilityTier"]
].copy()

validation_predictions["LogisticPrediction"] = lr_validation_predictions
validation_predictions["RandomForestPrediction"] = rf_validation_predictions

print("2023 Validation Predictions by Borough")
display(
    validation_predictions
    .sort_values("Borough")
    .reset_index(drop=True)
)

# Model-level validation summary
validation_results = pd.DataFrame([
    lr_validation_results,
    rf_validation_results
])

print("2023 Validation Performance Summary")
display(validation_results)

2023 Validation Predictions by Borough


,Borough,Year,FeasibilityTier,LogisticPrediction,RandomForestPrediction
0,Croydon,2023,Achievable,Stretch,Achievable
1,Kingston upon Thames,2023,Stretch,Stretch,Stretch
2,Merton,2023,Stretch,Stretch,Stretch
3,Richmond upon Thames,2023,Unfeasible,Stretch,Unfeasible
4,Sutton,2023,Stretch,Stretch,Stretch
5,Wandsworth,2023,Stretch,Stretch,Unfeasible


2023 Validation Performance Summary


,Model,Accuracy,MacroF1
0,Logistic Regression,0.666667,0.266667
1,Random Forest,0.833333,0.841270


### Validation comparison

In [69]:
# 2023 VALIDATION PREDICTIONS

lr_validation_predictions = logistic_model.predict(X_validation)
rf_validation_predictions = random_forest_model.predict(X_validation)

validation_predictions = validation_df[
    ["Borough", "Year", "FeasibilityTier"]
].copy()

validation_predictions["LogisticPrediction"] = lr_validation_predictions
validation_predictions["RandomForestPrediction"] = rf_validation_predictions


# 2023 VALIDATION METRICS

from sklearn.metrics import accuracy_score, f1_score

lr_validation_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_validation, lr_validation_predictions),
    "MacroF1": f1_score(
        y_validation,
        lr_validation_predictions,
        average="macro"
    )
}

rf_validation_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_validation, rf_validation_predictions),
    "MacroF1": f1_score(
        y_validation,
        rf_validation_predictions,
        average="macro"
    )
}


# 2023 VALIDATION COMPARISON

display(
    validation_predictions
    .sort_values("Borough")
    .reset_index(drop=True)
)

validation_results = pd.DataFrame([
    lr_validation_results,
    rf_validation_results
])

display(validation_results)

,Borough,Year,FeasibilityTier,LogisticPrediction,RandomForestPrediction
0,Croydon,2023,Achievable,Stretch,Achievable
1,Kingston upon Thames,2023,Stretch,Stretch,Stretch
2,Merton,2023,Stretch,Stretch,Stretch
3,Richmond upon Thames,2023,Unfeasible,Stretch,Unfeasible
4,Sutton,2023,Stretch,Stretch,Stretch
5,Wandsworth,2023,Stretch,Stretch,Unfeasible


,Model,Accuracy,MacroF1
0,Logistic Regression,0.666667,0.266667
1,Random Forest,0.833333,0.841270


### Validation Interpretation
Random Forest outperformed Logistic Regression on the 2023 validation period, achieving an accuracy of 0.833 and macro F1 of 0.841, compared with 0.667 and 0.267 respectively for Logistic Regression.
Although Logistic Regression correctly classified four of the six observations, it predicted every 2023 case as Stretch and therefore failed to identify both the Achievable and Unfeasible classes. Its accuracy consequently overstates its class-level performance.
Random Forest correctly classified five of the six boroughs and identified observations from all three feasibility classes. The only misclassification was Wandsworth, which was classified as Unfeasible rather than Stretch.
These results provide preliminary evidence in favour of Random Forest. However, the validation period contains only six observations, so the results are treated as a model-selection checkpoint rather than final performance evidence.


### Model comparison and selection criteria
This comparison asks whether the non-linear Random Forest provides enough improvement over Logistic Regression to justify its additional complexity.

* **Selected Model:** **Logistic Regression**.

* **Rationale:** On the final untouched 2024–2025 test set, Logistic Regression outperformed Random Forest, achieving 83.3% accuracy and a Macro F1 score of 0.778, compared with 58.3% accuracy and 0.444 Macro F1 for Random Forest. The additional complexity of Random Forest therefore did not result in stronger temporal generalisation.

* Random Forest performed better at the six-observation 2023 validation checkpoint, achieving 83.3% accuracy and 0.841 Macro F1 compared with 66.7% accuracy and 0.267 Macro F1 for Logistic Regression. However, this validation result was treated as preliminary because the sample contained only six observations.

* After validation, model settings were frozen and both models were refitted on the combined 2014–2023 development period. Final model selection was then based on the untouched 2024–2025 temporal test together with transparency and interpretability considerations.

* Logistic Regression was therefore selected for the FundFirst prototype because it showed stronger final temporal performance while retaining a simpler and more transparent model structure.

### Frozen model settings
No additional hyperparameter tuning is performed after this point.

In [70]:
RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

Following validation and fixation of model settings, both models were refitted on the combined 2014–2023 development data to maximise the limited training sample before a single final evaluation on the untouched 2024–2025 test period.

In [71]:
# REFIT USING 2014-2023
development_df = fundfirst[
    fundfirst["Year"].between(2014, 2023)
].copy()

X_development = development_df[FEATURES]
y_development = development_df[TARGET]

print("Development rows:", len(development_df))

print("\nDevelopment class distribution:")
print(y_development.value_counts())

Development rows: 60

Development class distribution:
FeasibilityTier
Stretch       25
Unfeasible    21
Achievable    14
Name: count, dtype: int64


In [72]:
logistic_final = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

logistic_final.fit(
    X_development,
    y_development
)

print("Final Logistic Regression fitted on 2014-2023.")

Final Logistic Regression fitted on 2014-2023.


In [73]:
import matplotlib.pyplot as plt
import numpy as np

models = ['Logistic Regression', 'Random Forest']
accuracy = [0.833, 0.583]
macro_f1 = [0.778, 0.444]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(7, 4.5))

plt.bar(x - width/2, accuracy, width, label='Accuracy')
plt.bar(x + width/2, macro_f1, width, label='Macro-F1')

plt.xticks(x, models)
plt.ylim(0, 1)
plt.ylabel('Score')
plt.title('Held-Out Model Performance, 2024 to 2025')
plt.legend()

for i, value in enumerate(accuracy):
    plt.text(i - width/2, value + 0.02, f'{value:.3f}', ha='center')

for i, value in enumerate(macro_f1):
    plt.text(i + width/2, value + 0.02, f'{value:.3f}', ha='center')

plt.tight_layout()
plt.savefig(
    FIGURES / "held_out_model_performance.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/1418565722.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [74]:
random_forest_final = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_forest_final.fit(
    X_development,
    y_development
)

print("Final Random Forest fitted on 2014-2023.")

Final Random Forest fitted on 2014-2023.


In [75]:
lr_test_results, lr_test_predictions = evaluate_model(
    logistic_final,
    X_test,
    y_test,
    "Logistic Regression"
)

rf_test_results, rf_test_predictions = evaluate_model(
    random_forest_final,
    X_test,
    y_test,
    "Random Forest"
)


Logistic Regression
----------------------------------------
Accuracy: 0.833
Macro F1: 0.778

Classification Report:
              precision    recall  f1-score   support

  Achievable       1.00      1.00      1.00         4
     Stretch       0.71      1.00      0.83         5
  Unfeasible       1.00      0.33      0.50         3

    accuracy                           0.83        12
   macro avg       0.90      0.78      0.78        12
weighted avg       0.88      0.83      0.81        12


Random Forest
----------------------------------------
Accuracy: 0.583
Macro F1: 0.444

Classification Report:
              precision    recall  f1-score   support

  Achievable       1.00      0.50      0.67         4
     Stretch       0.50      1.00      0.67         5
  Unfeasible       0.00      0.00      0.00         3

    accuracy                           0.58        12
   macro avg       0.50      0.50      0.44        12
weighted avg       0.54      0.58      0.50        12



In [76]:
test_results = pd.DataFrame([
    lr_test_results,
    rf_test_results
])

display(test_results)

,Model,Accuracy,MacroF1
0,Logistic Regression,0.833333,0.777778
1,Random Forest,0.583333,0.444444


In [77]:
test_predictions = test_df[
    [
        "Borough",
        "Year",
        "FeasibilityTier"
    ]
].copy()

test_predictions[
    "LogisticPrediction"
] = lr_test_predictions

test_predictions[
    "RandomForestPrediction"
] = rf_test_predictions

display(
    test_predictions.sort_values(
        ["Year", "Borough"]
    )
)

,Borough,Year,FeasibilityTier,LogisticPrediction,RandomForestPrediction
10,Croydon,2024,Achievable,Achievable,Achievable
22,Kingston upon Thames,2024,Stretch,Stretch,Stretch
34,Merton,2024,Stretch,Stretch,Stretch
46,Richmond upon Thames,2024,Unfeasible,Stretch,Stretch
58,Sutton,2024,Achievable,Achievable,Stretch
70,Wandsworth,2024,Unfeasible,Stretch,Stretch
11,Croydon,2025,Achievable,Achievable,Achievable
23,Kingston upon Thames,2025,Stretch,Stretch,Stretch
35,Merton,2025,Stretch,Stretch,Stretch
47,Richmond upon Thames,2025,Unfeasible,Unfeasible,Stretch


### Final Test Interpretation

Performance on the held-out 2024–2025 test period differed substantially from the 2023 validation results. Logistic Regression achieved an accuracy of 0.833 and macro F1 of 0.778, outperforming Random Forest, which achieved an accuracy of 0.583 and macro F1 of 0.444. Logistic Regression correctly classified all Achievable and Stretch observations, although recall for the Unfeasible class was lower at 0.33. Random Forest correctly identified all Stretch observations but
failed to identify any of the three Unfeasible cases and identified only half of the Achievable cases. The result reverses the model ranking observed during the six-observation 2023 validation period. This indicates that the single-year validation results were unstable and that the additional complexity of Random Forest did not translate into stronger temporal generalisation. On the final held-out period, the simpler Logistic Regression model reproduced the calibrated TSM classes more reliably. Given the small test sample of 12 observations, these findings should be interpreted as prototype-level evidence rather than as evidence of general performance across the wider London housing market. As a result, Logistic Regression was selected for the final prototype following completion of the comparative evaluation because it demonstrated stronger held-out temporal performance while also providing a simpler and more transparent model structure.

### Final Held-Out Test Visualisation

In [78]:
# Logistic Regression: 2024-2025 held-out confusion matrix
lr_test_display = ConfusionMatrixDisplay.from_predictions(
    y_test,
    lr_test_predictions,
    labels=CLASS_ORDER,
    xticks_rotation=30,
    cmap="Blues"
)

lr_test_display.ax_.set_title(
    "Logistic Regression - 2024-2025 Held-Out Test",
    fontsize=13,
    fontweight="bold"
)
lr_test_display.ax_.set_xlabel("Predicted Class")
lr_test_display.ax_.set_ylabel("Actual Class")

plt.tight_layout()
lr_test_display.figure_.savefig(
    FIGURES / "lr_test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/2887740285.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [79]:
# Random Forest: 2024-2025 held-out confusion matrix
rf_test_display = ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_test_predictions,
    labels=CLASS_ORDER,
    xticks_rotation=30,
    cmap="Blues"
)

rf_test_display.ax_.set_title(
    "Random Forest - 2024-2025 Held-Out Test",
    fontsize=13,
    fontweight="bold"
)
rf_test_display.ax_.set_xlabel("Predicted Class")
rf_test_display.ax_.set_ylabel("Actual Class")

plt.tight_layout()
rf_test_display.figure_.savefig(
    FIGURES / "rf_test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/3946625779.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [80]:
test_results.to_csv(
    OUTPUTS / "model_test_results.csv",
    index=False
)

test_predictions.to_csv(
    OUTPUTS / "test_predictions.csv",
    index=False
)

print("Final model results saved.")

Final model results saved.


In [81]:
print("Saved files:")

for file in OUTPUTS.iterdir():
    print(file.name)

Saved files:
test_predictions.csv
figures
model_test_results.csv


## 10. Final model selection 

* **Selected model:** **Logistic Regression**.
* **Held-out performance:** accuracy = **0.833**, macro F1 = **0.778**.
* **Random Forest comparison:** accuracy = **0.583**, macro F1 = **0.444**.
* **Rationale:** although Random Forest performed better on the six-row 2023 validation checkpoint, Logistic Regression generalised more strongly to the untouched 2024–2025 period and provides a simpler, more transparent structure for the prototype.

The test results are now treated as final and are not used for further tuning.

In [82]:
# FINAL MODEL SELECTION
selected_model_name = "Logistic Regression"
selected_model = logistic_final

print("Selected model:", selected_model_name)

Selected model: Logistic Regression


### Final Model Selection Rationale

Logistic Regression was selected as the final FundFirst prototype model.
Although Random Forest performed better during the six-observation 2023
validation period, Logistic Regression generalised more strongly to the
held-out 2024–2025 test period, achieving 0.833 accuracy and 0.778 macro
F1 compared with 0.583 and 0.444 for Random Forest.

The simpler model was therefore preferred because it demonstrated
stronger temporal generalisation while also supporting a more transparent
and interpretable model structure.

## 11. SHAP explainability

SHAP is used to explain the selected Logistic Regression model. `LinearExplainer` is applied to the fitted classifier using the already-fitted standardised feature representation.

### Explanation Coverage
All 12 held-out 2024–2025 predictions receive a local SHAP explanation.

### Example Local Explanation
One detailed held-out observation is shown with original, human-readable feature values. Dark blue contributions increase the model score for the predicted class; light blue contributions decrease it.

### Borough-Level Explanations
One 2025 local explanation is generated and saved for each of the six South West London boroughs.

### Global Feature Contributions
Mean absolute SHAP values are aggregated across held-out rows and classes to summarise the overall magnitude of each feature's contribution.

**Interpretation boundary:** SHAP explains how the trained model score is formed. It does not establish that a feature caused the housing outcome or that changing a feature will necessarily cause a different real-world outcome.

In [83]:
#  SHAP SETUP FOR LOGISTIC REGRESSION
scaler = logistic_final.named_steps["scaler"]
classifier = logistic_final.named_steps["classifier"]

In [84]:
# Features used by the model
FEATURES = [
    "AveragePrice",
    "MedianAnnualPay",
    "SavingRatio",
    "BaseRate"
]

# Recreate development and test feature sets
development_df = fundfirst[
    fundfirst["Year"].between(2014, 2023)
].copy()

test_df = fundfirst[
    fundfirst["Year"].between(2024, 2025)
].copy()

X_development = development_df[FEATURES]
X_test = test_df[FEATURES]

# Extract fitted scaler and classifier
scaler = logistic_final.named_steps["scaler"]
classifier = logistic_final.named_steps["classifier"]

# Use the already-fitted scaler
X_development_scaled = scaler.transform(X_development)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames so SHAP keeps feature names
X_development_scaled_df = pd.DataFrame(
    X_development_scaled,
    columns=FEATURES,
    index=X_development.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=FEATURES,
    index=X_test.index
)

# Create SHAP explainer
shap_explainer = shap.LinearExplainer(
    classifier,
    X_development_scaled_df
)

shap_values = shap_explainer(
    X_test_scaled_df
)

print("SHAP values shape:", shap_values.values.shape)
print("Number of test rows:", len(X_test))
print("Model classes:", classifier.classes_)

SHAP values shape: (12, 4, 3)
Number of test rows: 12
Model classes: ['Achievable' 'Stretch' 'Unfeasible']


In [85]:
assert len(shap_values.values) == len(X_test)

print(
    "Every 2024-2025 test prediction has a SHAP explanation."
)

Every 2024-2025 test prediction has a SHAP explanation.


In [86]:
# Example local SHAP explanation for the first held-out observation
row_number = 0
row_info = test_df.iloc[row_number]

predicted_class = logistic_final.predict(
    X_test.iloc[[row_number]]
)[0]

class_index = list(classifier.classes_).index(predicted_class)
local_shap = shap_values.values[row_number, :, class_index]
feature_values = X_test.iloc[row_number].values

feature_labels = [
    f"Average Price (£{feature_values[0]:,.0f})",
    f"Median Annual Pay (£{feature_values[1]:,.0f})",
    f"Saving Ratio ({feature_values[2]:.1f}%)",
    f"Base Rate ({feature_values[3]:.2f}%)"
]

order = np.argsort(np.abs(local_shap))
sorted_values = local_shap[order]
sorted_labels = np.array(feature_labels)[order]

# Dark blue increases the predicted-class score; light blue decreases it.
colors = [
    "#1f4e79" if value >= 0 else "#9dc3e6"
    for value in sorted_values
]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    sorted_labels,
    sorted_values,
    color=colors,
    edgecolor="#244062",
    linewidth=0.8
)

ax.axvline(0, color="black", linewidth=0.8)

for bar, value in zip(bars, sorted_values):
    offset = 0.02 if value >= 0 else -0.02
    ax.text(
        value + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{value:+.2f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=10
    )

ax.set_title(
    f"Local SHAP Explanation: {row_info['Borough']} ({int(row_info['Year'])})\n"
    f"Predicted Tier: {predicted_class}",
    fontsize=13,
    fontweight="bold",
    pad=12
)
ax.set_xlabel(f"Contribution to '{predicted_class}' model score", fontsize=11)
ax.set_ylabel("")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()

example_name = (
    f"shap_local_{row_info['Borough'].lower().replace(' ', '_')}_"
    f"{int(row_info['Year'])}.png"
)
fig.savefig(FIGURES / example_name, dpi=300, bbox_inches="tight")
plt.show()

print("Saved example SHAP figure:", example_name)

Saved example SHAP figure: shap_local_croydon_2024.png


/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/2796343296.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [87]:
# Select one 2025 held-out observation for each borough
borough_2025 = test_df[
    test_df["Year"] == 2025
].copy()

display(
    borough_2025[
        ["Borough", "Year", "FeasibilityTier"]
    ].sort_values("Borough")
)

,Borough,Year,FeasibilityTier
11,Croydon,2025,Achievable
23,Kingston upon Thames,2025,Stretch
35,Merton,2025,Stretch
47,Richmond upon Thames,2025,Unfeasible
59,Sutton,2025,Achievable
71,Wandsworth,2025,Stretch


In [88]:
# Generate and save one local SHAP plot for each borough in 2025
for idx in borough_2025.index:
    row_number = list(X_test.index).index(idx)
    row_info = test_df.loc[idx]

    predicted_class = logistic_final.predict(
        X_test.loc[[idx]]
    )[0]
    class_index = list(classifier.classes_).index(predicted_class)

    local_shap = shap_values.values[row_number, :, class_index]
    feature_values = X_test.loc[idx].values

    feature_labels = [
        f"Average Price (£{feature_values[0]:,.0f})",
        f"Median Annual Pay (£{feature_values[1]:,.0f})",
        f"Saving Ratio ({feature_values[2]:.1f}%)",
        f"Base Rate ({feature_values[3]:.2f}%)"
    ]

    order = np.argsort(np.abs(local_shap))
    sorted_values = local_shap[order]
    sorted_labels = np.array(feature_labels)[order]

    colors = [
        "#1f4e79" if value >= 0 else "#9dc3e6"
        for value in sorted_values
    ]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.barh(
        sorted_labels,
        sorted_values,
        color=colors,
        edgecolor="#244062",
        linewidth=0.8
    )

    ax.axvline(0, color="black", linewidth=0.8)

    for bar, value in zip(bars, sorted_values):
        offset = 0.02 if value >= 0 else -0.02
        ax.text(
            value + offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value:+.2f}",
            va="center",
            ha="left" if value >= 0 else "right",
            fontsize=9
        )

    ax.set_title(
        f"{row_info['Borough']} - 2025\nPredicted: {predicted_class}",
        fontsize=12,
        fontweight="bold"
    )
    ax.set_xlabel(f"Contribution to '{predicted_class}' model score")
    ax.set_ylabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

    plt.tight_layout()

    safe_borough = row_info["Borough"].lower().replace(" ", "_")
    figure_path = FIGURES / f"shap_local_{safe_borough}_2025.png"
    fig.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.show()

print("Six borough-level 2025 SHAP figures saved.")

/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/428422206.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/428422206.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/428422206.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/428422206.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/428422206.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Six borough-level 2025 SHAP figures saved.


/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/428422206.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Global SHAP Feature Contributions

This summary reports the mean absolute SHAP contribution for each input feature across all held-out observations and all three model classes. It describes contribution magnitude, not causal importance.

In [89]:
# Global SHAP feature-contribution summary
mean_abs_shap = np.abs(shap_values.values).mean(axis=(0, 2))

global_shap = pd.DataFrame({
    "Feature": FEATURES,
    "MeanAbsoluteSHAP": mean_abs_shap
}).sort_values(
    "MeanAbsoluteSHAP",
    ascending=True
)

display(global_shap.sort_values("MeanAbsoluteSHAP", ascending=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(
    global_shap["Feature"],
    global_shap["MeanAbsoluteSHAP"],
    color="#1f4e79",
    edgecolor="#244062",
    linewidth=0.8
)
ax.set_title(
    "Global SHAP Feature Contribution - Logistic Regression",
    fontsize=13,
    fontweight="bold"
)
ax.set_xlabel("Mean Absolute SHAP Value")
ax.set_ylabel("")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
fig.savefig(
    FIGURES / "shap_global_feature_contribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Saved figures:")
for file in sorted(FIGURES.glob("*.png")):
    print(file.name)

,Feature,MeanAbsoluteSHAP
0,AveragePrice,1.646138
1,MedianAnnualPay,1.120753
3,BaseRate,0.883323
2,SavingRatio,0.061117


Saved figures:
held_out_model_performance.png
lr_test_confusion_matrix.png
lr_validation_confusion_matrix.png
rf_test_confusion_matrix.png
rf_validation_confusion_matrix.png
shap_global_feature_contribution.png
shap_local_croydon_2024.png
shap_local_croydon_2025.png
shap_local_kingston_upon_thames_2025.png
shap_local_merton_2025.png
shap_local_richmond_upon_thames_2025.png
shap_local_sutton_2025.png
shap_local_wandsworth_2025.png


/var/folders/m4/cxxk1wxj30g0rq4pjqf6xhc80000gn/T/ipykernel_12092/900414045.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Income-stratified performance audit

The selected Logistic Regression model is audited across income-stratified borough groups using development-period median annual pay as a transparent proxy for socioeconomic variation.

* **Groups:** Higher Income vs Lower Income boroughs, defined using the median of borough-level mean development-period income.
* **Metrics:** overall accuracy and per-class precision.
* **Fairness flag threshold:** any **estimable** group gap above 10 percentage points is flagged for discussion.
* If a group has no predictions for a class, that class precision is reported as `NaN` (not estimable), rather than incorrectly treating it as zero.

A flagged gap is evidence of a performance disparity requiring discussion; it does not by itself establish the cause of the disparity or prove unfair treatment.


In [90]:
# INCOME BIAS AUDIT

# Mean income for each borough using development data only
borough_income = (
    development_df
    .groupby("Borough", as_index=False)
    .agg(
        MeanDevelopmentIncome=(
            "MedianAnnualPay",
            "mean"
        )
    )
)

# Median borough income used as grouping threshold
income_threshold = (
    borough_income[
        "MeanDevelopmentIncome"
    ].median()
)

borough_income["IncomeGroup"] = np.where(
    borough_income["MeanDevelopmentIncome"]
    >= income_threshold,
    "Higher Income",
    "Lower Income"
)

display(borough_income)

print(
    f"Income grouping threshold: "
    f"£{income_threshold:,.2f}"
)

,Borough,MeanDevelopmentIncome,IncomeGroup
0,Croydon,34126.80,Higher Income
1,Kingston upon Thames,33013.85,Lower Income
2,Merton,31681.50,Lower Income
3,Richmond upon Thames,34671.70,Higher Income
4,Sutton,30844.30,Lower Income
5,Wandsworth,34335.70,Higher Income


Income grouping threshold: £33,570.32


In [91]:
audit_df = test_predictions.merge(
    borough_income[
        [
            "Borough",
            "IncomeGroup"
        ]
    ],
    on="Borough",
    how="left"
)

audit_df["Prediction"] = (
    audit_df["LogisticPrediction"]
)

audit_df["Correct"] = (
    audit_df["Prediction"]
    ==
    audit_df["FeasibilityTier"]
)

display(audit_df)

,Borough,Year,FeasibilityTier,LogisticPrediction,RandomForestPrediction,IncomeGroup,Prediction,Correct
0,Croydon,2024,Achievable,Achievable,Achievable,Higher Income,Achievable,True
1,Croydon,2025,Achievable,Achievable,Achievable,Higher Income,Achievable,True
2,Kingston upon Thames,2024,Stretch,Stretch,Stretch,Lower Income,Stretch,True
3,Kingston upon Thames,2025,Stretch,Stretch,Stretch,Lower Income,Stretch,True
4,Merton,2024,Stretch,Stretch,Stretch,Lower Income,Stretch,True
5,Merton,2025,Stretch,Stretch,Stretch,Lower Income,Stretch,True
6,Richmond upon Thames,2024,Unfeasible,Stretch,Stretch,Higher Income,Stretch,False
7,Richmond upon Thames,2025,Unfeasible,Unfeasible,Stretch,Higher Income,Unfeasible,True
8,Sutton,2024,Achievable,Achievable,Stretch,Lower Income,Achievable,True
9,Sutton,2025,Achievable,Achievable,Stretch,Lower Income,Achievable,True


In [92]:
group_accuracy = (
    audit_df
    .groupby(
        "IncomeGroup",
        as_index=False
    )
    .agg(
        Accuracy=("Correct", "mean"),
        NumberOfCases=("Correct", "size")
    )
)

group_accuracy["Accuracy"] = (
    group_accuracy["Accuracy"]
    .round(3)
)

display(group_accuracy)

,IncomeGroup,Accuracy,NumberOfCases
0,Higher Income,0.667,6
1,Lower Income,1.000,6


In [93]:
accuracy_gap = (
    group_accuracy["Accuracy"].max()
    -
    group_accuracy["Accuracy"].min()
)

print(
    f"Accuracy gap: "
    f"{accuracy_gap * 100:.1f} percentage points"
)

FAIRNESS_THRESHOLD = 0.10

if accuracy_gap > FAIRNESS_THRESHOLD:
    print(
        "FLAG: Accuracy gap exceeds "
        "10 percentage points."
    )
else:
    print(
        "No accuracy gap above "
        "10 percentage points."
    )

Accuracy gap: 33.3 percentage points
FLAG: Accuracy gap exceeds 10 percentage points.


In [94]:
precision_rows = []

for group_name, group_df in audit_df.groupby(
    "IncomeGroup"
):

    for class_name in CLASS_ORDER:

        predicted_class = (
            group_df["Prediction"]
            == class_name
        )

        predicted_count = (
            predicted_class.sum()
        )

        true_positive = (
            predicted_class
            &
            (
                group_df["FeasibilityTier"]
                == class_name
            )
        ).sum()

        if predicted_count == 0:
            precision = np.nan
        else:
            precision = (
                true_positive /
                predicted_count
            )

        precision_rows.append({
            "IncomeGroup": group_name,
            "Class": class_name,
            "Precision": precision,
            "PredictedCount": predicted_count
        })

precision_audit = pd.DataFrame(
    precision_rows
)

display(precision_audit)

,IncomeGroup,Class,Precision,PredictedCount
0,Higher Income,Achievable,1.000000,2
1,Higher Income,Stretch,0.333333,3
2,Higher Income,Unfeasible,1.000000,1
3,Lower Income,Achievable,1.000000,2
4,Lower Income,Stretch,1.000000,4
5,Lower Income,Unfeasible,NaN,0


In [95]:
# Compare per-class precision between the two income groups.
# If either group has undefined precision for a class, the gap remains NaN.
precision_pivot = precision_audit.pivot(
    index="Class",
    columns="IncomeGroup",
    values="Precision"
)

precision_pivot["PrecisionGap"] = (
    precision_pivot["Higher Income"]
    - precision_pivot["Lower Income"]
).abs()

precision_pivot["GapPercentagePoints"] = (
    precision_pivot["PrecisionGap"] * 100
)

precision_pivot["FlagOver10pp"] = (
    precision_pivot["PrecisionGap"].notna()
    & (precision_pivot["PrecisionGap"] > FAIRNESS_THRESHOLD)
)

display(precision_pivot.round(3))

IncomeGroup,Higher Income,Lower Income,PrecisionGap,GapPercentagePoints,FlagOver10pp
Class,,,,,
Achievable,1.000,1.0,0.000,0.000,False
Stretch,0.333,1.0,0.667,66.667,True
Unfeasible,1.000,NaN,NaN,NaN,False


In [96]:
# Report estimable fairness flags and explicitly identify undefined gaps
for class_name, row in precision_pivot.iterrows():
    if pd.isna(row["PrecisionGap"]):
        print(
            f"{class_name}: precision gap not estimable because at least one "
            "income group has no predictions for this class."
        )
    elif row["FlagOver10pp"]:
        print(
            f"FLAG - {class_name}: {row['GapPercentagePoints']:.1f} percentage-point "
            "precision gap exceeds the 10pp threshold."
        )
    else:
        print(
            f"{class_name}: {row['GapPercentagePoints']:.1f} percentage-point "
            "precision gap does not exceed the 10pp threshold."
        )

Achievable: 0.0 percentage-point precision gap does not exceed the 10pp threshold.
FLAG - Stretch: 66.7 percentage-point precision gap exceeds the 10pp threshold.
Unfeasible: precision gap not estimable because at least one income group has no predictions for this class.


In [97]:
group_accuracy.to_csv(
    OUTPUTS / "bias_audit_accuracy.csv",
    index=False
)

precision_audit.to_csv(
    OUTPUTS / "bias_audit_precision.csv",
    index=False
)

precision_pivot.to_csv(
    OUTPUTS / "bias_audit_precision_gaps.csv"
)

print("Bias audit outputs saved.")

Bias audit outputs saved.


### Fairness Audit Interpretation

On the current 2024–2025 held-out sample, Higher Income boroughs achieve 0.667 accuracy and Lower Income boroughs 1.000 accuracy, giving a **33.3 percentage-point accuracy gap**, which exceeds the predefined 10pp flag threshold.

For per-class precision, the **Stretch** class has an estimable **66.7 percentage-point gap** (Higher Income = 0.333; Lower Income = 1.000), which is also flagged. Achievable precision shows no gap. The Unfeasible precision gap is **not estimable** because the Lower Income group has no predicted Unfeasible cases.

These disparities are reported as fairness flags requiring cautious discussion. Given the 12-row held-out sample and borough-level aggregation, the audit cannot establish whether the gaps arise from model behaviour, differences in borough profiles, or sampling variation. Income is used only as a socioeconomic proxy and is not a substitute for protected-characteristic fairness assessment.

## 13. Model card generation

A structured model card is generated from the final evaluation, explainability, and fairness-audit outputs so that the documented evidence remains consistent with the executed notebook.

In [98]:
# MODEL CARD METRICS AND FAIRNESS SUMMARY
MODEL_CARD = PROJECT_ROOT / "model_card.md"

lr_accuracy = test_results.loc[
    test_results["Model"] == "Logistic Regression",
    "Accuracy"
].iloc[0]

lr_macro_f1 = test_results.loc[
    test_results["Model"] == "Logistic Regression",
    "MacroF1"
].iloc[0]

rf_accuracy = test_results.loc[
    test_results["Model"] == "Random Forest",
    "Accuracy"
].iloc[0]

rf_macro_f1 = test_results.loc[
    test_results["Model"] == "Random Forest",
    "MacroF1"
].iloc[0]

flagged_precision = precision_pivot[
    precision_pivot["FlagOver10pp"]
]

if flagged_precision.empty:
    precision_flag_text = "No estimable per-class precision gap exceeded 10 percentage points."
else:
    precision_flag_text = "; ".join(
        f"{class_name}: {row['GapPercentagePoints']:.1f}pp"
        for class_name, row in flagged_precision.iterrows()
    )

undefined_precision_classes = precision_pivot[
    precision_pivot["PrecisionGap"].isna()
].index.tolist()

undefined_precision_text = (
    ", ".join(undefined_precision_classes)
    if undefined_precision_classes
    else "None"
)

In [99]:
# Generate the FundFirst model card
model_card_text = f"""
# FundFirst Model Card

## Model Overview

FundFirst is an explainable multiclass classification prototype for
first-time buyer deposit feasibility across six South West London boroughs.

Selected model: Logistic Regression

Feasibility classes:
- Achievable
- Stretch
- Unfeasible

## Intended Purpose

FundFirst is intended to provide educational decision support by helping
prospective first-time buyers understand the relative feasibility of a
deposit-saving scenario.

It is not a mortgage approval, creditworthiness, lending, investment,
or regulated financial-advice system.

## Geographic Scope

- Wandsworth
- Richmond upon Thames
- Kingston upon Thames
- Merton
- Sutton
- Croydon

## Data Period

2014-2025.

Training and model development:
2014-2023.

Final held-out evaluation:
2024-2025.

## Model Features

- AveragePrice
- MedianAnnualPay
- SavingRatio
- BaseRate

## Label Generation

The target classes were generated using the transparent TSM rule.

A 10% deposit target and 20% calibrated active-saving assumption were
used to estimate MonthsToSave.

The final operational thresholds were:

- Achievable: <= 72 months
- Stretch: > 72 and <= 108 months
- Unfeasible: > 108 months

The 72-month Achievable boundary was selected following sensitivity
analysis on the 2014-2022 calibration period after the original
48-month threshold produced no Achievable observations.

## Model Comparison

Logistic Regression:
- Test accuracy: {lr_accuracy:.3f}
- Test macro F1: {lr_macro_f1:.3f}

Random Forest:
- Test accuracy: {rf_accuracy:.3f}
- Test macro F1: {rf_macro_f1:.3f}

Logistic Regression was selected because it demonstrated stronger
performance on the held-out 2024-2025 period while retaining a simpler
and more transparent structure.

## Explainability

Local SHAP explanations were generated for every held-out prediction.
A global mean-absolute-SHAP summary is also generated across held-out rows
and classes. SHAP values describe contributions to the model score and
should not be interpreted as causal effects.

## Fairness Audit

The model was evaluated across higher-income and lower-income borough
groups using development-period median earnings as a socioeconomic proxy.

Overall accuracy gap: {accuracy_gap * 100:.1f} percentage points.

Estimable per-class precision gaps above 10pp: {precision_flag_text}

Classes with a non-estimable precision gap because at least one group had
no predictions for that class: {undefined_precision_text}

The fairness audit is indicative only because the source data are
aggregated and contain no individual protected-characteristic data.

## Known Limitations

- The dataset contains only six boroughs and 72 borough-year observations.
- Labels are generated by a deterministic rule rather than observed
  home-purchase outcomes.
- MedianAnnualPay is an aggregate workplace-based borough-level measure.
- SavingRatio and BaseRate are national rather than borough-specific.
- The income-based fairness audit does not represent a comprehensive
  protected-characteristic fairness assessment.
- The held-out test set contains only 12 observations.
- Performance therefore represents prototype-level evidence rather than
  general evidence for all London first-time buyers.

## Governance boundary

The EU AI Act, FCA guidance and UK AI assurance principles are used as
design and evaluation guidance only.

The project does not claim legal compliance or constitute a formal AI
prototype risk review.

## Financial Disclaimer

FundFirst is an educational research prototype and does not provide
regulated financial, mortgage, investment or lending advice. Users should
consult an FCA-regulated professional before making financial commitments.
"""

MODEL_CARD.write_text(
    model_card_text,
    encoding="utf-8"
)

print("Model card saved:")
print(MODEL_CARD.relative_to(PROJECT_ROOT))

Model card saved:
/Users/sarah/Documents/Codex/2026-08-15/referenced-chatgpt-conversation-this-is-an/FundFirst_Code_Submission_260010319/model_card.md


## 14. Model serialisation

The selected fitted Logistic Regression pipeline is saved as a single `joblib` artifact. Because the `StandardScaler` is already contained inside the fitted pipeline, a separate scaler file is not required.

In [100]:
# MODEL SERIALISATION
MODEL_PATH = MODELS / "fundfirst_logistic_regression.joblib"

joblib.dump(
    logistic_final,
    MODEL_PATH
)

print("Selected model saved:")
print(MODEL_PATH.relative_to(PROJECT_ROOT))

Selected model saved:
/Users/sarah/Documents/Codex/2026-08-15/referenced-chatgpt-conversation-this-is-an/FundFirst_Code_Submission_260010319/models/fundfirst_logistic_regression.joblib


In [101]:
# Save model metadata for the prototype application
model_metadata = {
    "model_name": "FundFirst Logistic Regression",
    "selected_model": "Logistic Regression",
    "features": FEATURES,
    "classes": CLASS_ORDER,
    "achievable_max_months": ACHIEVABLE_MAX,
    "stretch_max_months": STRETCH_MAX,
    "deposit_rate": DEPOSIT_RATE,
    "active_saving_rate": ACTIVE_SAVING_RATE,
    "development_period": "2014-2023",
    "test_period": "2024-2025",
    "test_accuracy": float(lr_accuracy),
    "test_macro_f1": float(lr_macro_f1),
    "disclaimer": (
        "FundFirst is an educational research prototype and does not "
        "provide regulated financial advice."
    )
}

METADATA_PATH = MODELS / "fundfirst_model_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=4)

print("Model metadata saved:")
print(METADATA_PATH.relative_to(PROJECT_ROOT))

Model metadata saved:
/Users/sarah/Documents/Codex/2026-08-15/referenced-chatgpt-conversation-this-is-an/FundFirst_Code_Submission_260010319/models/fundfirst_model_metadata.json


In [102]:
# Verify that the serialised pipeline reloads and reproduces a prediction
loaded_model = joblib.load(MODEL_PATH)

original_prediction = logistic_final.predict(
    X_test.iloc[[0]]
)[0]

loaded_prediction = loaded_model.predict(
    X_test.iloc[[0]]
)[0]

print("Original prediction:", original_prediction)
print("Reloaded prediction:", loaded_prediction)

assert original_prediction == loaded_prediction

print("\nModel serialization check PASSED.")

Original prediction: Achievable
Reloaded prediction: Achievable

Model serialization check PASSED.


In [103]:
print("Saved model files:")
for file in sorted(MODELS.iterdir()):
    print(file.name)

Saved model files:
fundfirst_logistic_regression.joblib
fundfirst_model_metadata.json


## 15. Manual inference example

In [104]:
# Edit these four values to explore a new illustrative scenario.
test_average_price = 425000
test_annual_pay = 42000
test_saving_ratio = 9.6
test_base_rate = 4.25

new_scenario = pd.DataFrame({
    "AveragePrice": [test_average_price],
    "MedianAnnualPay": [test_annual_pay],
    "SavingRatio": [test_saving_ratio],
    "BaseRate": [test_base_rate]
})

display(new_scenario)

prediction = loaded_model.predict(new_scenario)[0]
probabilities = loaded_model.predict_proba(new_scenario)[0]
model_classes = loaded_model.named_steps["classifier"].classes_

print("FundFirst Prediction:", prediction)
print("\nPredicted Probabilities:")
for class_name, probability in zip(model_classes, probabilities):
    print(f"{class_name}: {probability:.1%}")

# Basic functional checks
assert prediction in CLASS_ORDER
assert len(probabilities) == 3
assert np.isclose(probabilities.sum(), 1.0)
assert list(new_scenario.columns) == FEATURES

print("\nAll inference checks PASSED.")

,AveragePrice,MedianAnnualPay,SavingRatio,BaseRate
0,425000,42000,9.6,4.25


FundFirst Prediction: Achievable

Predicted Probabilities:
Achievable: 85.3%
Stretch: 14.7%
Unfeasible: 0.0%

All inference checks PASSED.


In [105]:
# Transparent TSM reference calculation for the same manual scenario
# SavingRatio and BaseRate are model context features; they are not used in this rule calculation.
deposit_target = test_average_price * DEPOSIT_RATE
monthly_income = test_annual_pay / 12
monthly_savings = monthly_income * ACTIVE_SAVING_RATE
months_to_save = deposit_target / monthly_savings
years_to_save = months_to_save / 12

if months_to_save <= ACHIEVABLE_MAX:
    tsm_tier = "Achievable"
elif months_to_save <= STRETCH_MAX:
    tsm_tier = "Stretch"
else:
    tsm_tier = "Unfeasible"

print(f"10% deposit target: £{deposit_target:,.0f}")
print(f"Estimated monthly saving at 20% assumption: £{monthly_savings:,.0f}")
print(f"Estimated saving horizon: {months_to_save:.1f} months ({years_to_save:.1f} years)")
print(f"TSM reference tier: {tsm_tier}")
print(f"ML prediction: {prediction}")

10% deposit target: £42,500


Estimated monthly saving at 20% assumption: £700
Estimated saving horizon: 60.7 months (5.1 years)
TSM reference tier: Achievable
ML prediction: Achievable


## 16. Project summary

The final cell summarises the frozen FundFirst analytical pipeline and its held-out results.

In [106]:
print("=" * 62)
print("FUNDFIRST - FINAL PROJECT SUMMARY")
print("=" * 62)

print("\nDataset:")
print(f"Borough-year observations: {len(fundfirst)}")
print("Period: 2014-2025")

print("\nSelected model:")
print(selected_model_name)

print("\nHeld-out evaluation period:")
print("2024-2025")

print("\nFinal test performance:")
print(f"Logistic Regression accuracy: {lr_accuracy:.3f}")
print(f"Logistic Regression macro F1: {lr_macro_f1:.3f}")
print(f"Random Forest accuracy: {rf_accuracy:.3f}")
print(f"Random Forest macro F1: {rf_macro_f1:.3f}")

print("\nCalibrated TSM thresholds:")
print(f"Achievable: <= {ACHIEVABLE_MAX} months")
print(f"Stretch: > {ACHIEVABLE_MAX} and <= {STRETCH_MAX} months")
print(f"Unfeasible: > {STRETCH_MAX} months")

print("\nExplainability:")
print(f"Local SHAP coverage: {len(shap_values.values)}/{len(X_test)} held-out predictions")
print("Global SHAP summary: generated and saved")

print("\nFairness audit:")
print(f"Overall income-group accuracy gap: {accuracy_gap * 100:.1f} percentage points")
for class_name, row in precision_pivot.iterrows():
    if pd.isna(row["GapPercentagePoints"]):
        print(f"{class_name} precision gap: not estimable")
    else:
        status = "FLAG" if row["FlagOver10pp"] else "within threshold"
        print(f"{class_name} precision gap: {row['GapPercentagePoints']:.1f}pp ({status})")

print("\nSaved model:")
print(MODEL_PATH.relative_to(PROJECT_ROOT))
print("\nFundFirst analytical pipeline complete.")

FUNDFIRST - FINAL PROJECT SUMMARY

Dataset:
Borough-year observations: 72
Period: 2014-2025

Selected model:
Logistic Regression

Held-out evaluation period:
2024-2025

Final test performance:
Logistic Regression accuracy: 0.833
Logistic Regression macro F1: 0.778
Random Forest accuracy: 0.583
Random Forest macro F1: 0.444

Calibrated TSM thresholds:
Achievable: <= 72 months
Stretch: > 72 and <= 108 months
Unfeasible: > 108 months

Explainability:
Local SHAP coverage: 12/12 held-out predictions
Global SHAP summary: generated and saved

Fairness audit:
Overall income-group accuracy gap: 33.3 percentage points
Achievable precision gap: 0.0pp (within threshold)
Stretch precision gap: 66.7pp (FLAG)
Unfeasible precision gap: not estimable

Saved model:
/Users/sarah/Documents/Codex/2026-08-15/referenced-chatgpt-conversation-this-is-an/FundFirst_Code_Submission_260010319/models/fundfirst_logistic_regression.joblib

FundFirst analytical pipeline complete.
